# PleiadesConfig

`PleiadesConfig` is the backbone configuration object for PLEIADES. It
captures **what to run**, **where to run it**, and **what inputs/results
belong to each fit**. The goal is to keep a single, reproducible record
of the workflow so a run can be re-created later from the same config.

In practice it: 
- Defines the workspace directory layout used by SAMMY runs.
- Records nuclear inputs (isotopes + ENDF libraries).
- Declares datasets and fit routines to run.
- Serves as the source of record for a given execution of SAMMY with PLEIADES.
- Tracks run metadata and result locations.


## Create a PleiadesConfig from scratch in Python

Below we build a small PleiadesConfig with a workspace, a single isotope (Ta-181), and
the infomation of the fit routine.

This has the following substructures 
#### PleiadesConfig (overall)
- A working record of a given PLEIADES run: workspace layout, nuclear inputs, datasets, fit routines, results and metadata are all stored.
- Designed to make runs reproducible by keeping config + outputs + run metadata together.


#### WorkspaceConfig
- `root`: top-level workspace folder for all run artifacts.
- `endf_dir`: cache location for ENDF resonance files (defaults to the nuclear cache).
- `fitting_dir`: per‑routine working directories where `SAMMY` is executed.
- `results_dir`: top‑level results folder for aggregated outputs across run routines.
- `data_dir`: location for input data files (e.g., transmission `.dat`/`.twenty`).
- `image_dir`: optional location where the energy-resolved neutron imaging data is stored.

#### NuclearConfig
- `data_cache_dir`: root cache for ENDF resonance files (defaults to ~/.pleiades/nuclear_data).
- `sources`: URLs for ENDF retrieval (DIRECT/API).
- `default_library`: default ENDF library if not specified per isotope.
- `isotopes`: list of isotope entries to seed runs; used to populate fit_config.nuclear_params.isotopes (unless overridden per routine).

#### SammyConfig



In [ ]:
from pathlib import Path

from pleiades.utils.config import PleiadesConfig, WorkspaceConfig, NuclearConfig

# Workspace paths for generated files and SAMMY runs
workspace = WorkspaceConfig(
    root=Path("/tmp/pleiades_workspace"),
    fitting_dir=Path("/tmp/pleiades_workspace/fitting_dir"),
    results_dir=Path("/tmp/pleiades_workspace/results_dir"),
    data_dir=Path("/tmp/pleiades_workspace/data_dir"),
    image_dir=Path("/tmp/pleiades_workspace/image_dir"),
)

# Minimal nuclear configuration with one isotope
nuclear = NuclearConfig(
    isotopes=[
        {"isotope": "Ta-181", "abundance": 0.016, "vary_abundance": True},
    ]
)

# Fit routines must be present when loading from YAML
config = PleiadesConfig(
    pleiades_version=1,
    workspace=workspace,
    nuclear=nuclear,
    fit_routines={"example_fit": {"dataset_id": "example_dataset"}},
)


pleiades_version=1 workspace=WorkspaceConfig(root=PosixPath('/tmp/pleiades_workspace'), endf_dir=PosixPath('/Users/alexlong/.pleiades/nuclear_data'), fitting_dir=PosixPath('/tmp/pleiades_workspace/fitting_dir'), results_dir=PosixPath('/tmp/pleiades_workspace/results_dir'), data_dir=PosixPath('/tmp/pleiades_workspace/data_dir'), image_dir=PosixPath('/tmp/pleiades_workspace/image_dir')) nuclear=NuclearConfig(data_cache_dir=PosixPath('/Users/alexlong/.pleiades/nuclear_data'), sources={'DIRECT': 'https://www-nds.iaea.org/public/download-endf', 'API': 'https://www-nds.iaea.org/exfor/servlet'}, default_library=<EndfLibrary.ENDF_B_VIII_0: 'ENDF-B-VIII.0'>, isotopes=[IsotopeConfig(isotope='Ta-181', abundance=0.016, uncertainty=None, vary_abundance=<VaryFlag.YES: 1>, endf_library=<EndfLibrary.ENDF_B_VIII_0: 'ENDF-B-VIII.0'>)]) sammy=None datasets={} fit_routines={'example_fit': {'dataset_id': 'example_dataset'}} runs=[] results_index={} nuclear_data_cache_dir=PosixPath('/Users/alexlong/.pleiade

## Export to a YAML-friendly dictionary

`to_dict()` converts the Pydantic model into plain Python values that
can be serialized to YAML.


In [ ]:
config.to_dict()


## Load from YAML

The example YAML file lives next to this notebook at
`examples/Notebooks/getting_started/pleiades_config.yaml`.
Loading uses the same validation rules as the Python model.


In [ ]:
from pleiades.utils.config import PleiadesConfig

yaml_path = Path("examples/Notebooks/getting_started/pleiades_config.yaml")
loaded_config = PleiadesConfig.load(yaml_path)

loaded_config


## Notes

- `fit_routines` is required when loading from YAML.
- `workspace.endf_dir` defaults to the nuclear data cache if omitted.
- Isotopes are normalized so a missing `endf_library` defaults to ENDF-B-VIII.0.
